# Day 03 — Async/Await Fundamentals

**Phase 1 · Module 1: LangGraph State** · ~30 min

### What I practice today
- [ ] Convert a **sync** LangGraph node function to **`async def`**
- [ ] Use **`await`** on an LLM call inside your node
- [ ] Test with **`asyncio.run()`** locally

### Why this matters
An agent spends almost all its wall-clock time **waiting** — for an LLM to answer, for a tool's HTTP call, for a database. That waiting is **I/O**, and while one node waits, the CPU is idle. `async`/`await` lets a node say *"I'm waiting — go run something else meanwhile."* When a graph fans out to several LLM/tool calls, async turns *sum of the waits* into *the single longest wait*.

> Rule of thumb: *`async` doesn't make one call faster — it stops other work from blocking while that call waits.*

## 1. The problem async solves — blocking I/O

A normal (sync) function that waits **blocks the whole thread**: nothing else can run until it returns. Simulate two "LLM calls" that each take 1 second with `time.sleep` and watch them run **one after another**.

In [1]:
import time

def sync_llm(prompt: str) -> str:
    time.sleep(1)                 # stand-in for a slow network/LLM call
    return f"answer to: {prompt}"

start = time.perf_counter()
a = sync_llm("question 1")
b = sync_llm("question 2")        # cannot start until the first fully returns
print(a, "|", b)
print(f"took {time.perf_counter() - start:.1f}s")   # ~2.0s -- the waits ADD UP

answer to: question 1 | answer to: question 2
took 2.0s


Two 1-second waits took ~2s because the second call couldn't begin until the first finished. The CPU sat idle the whole time. Async fixes exactly this.

In [2]:
# Preview: the SAME two 1-second calls, but run CONCURRENTLY with async.
# (The keywords used here -- async def / await / gather -- are explained in sections 2-4.)
import asyncio

async def async_llm_preview(prompt: str) -> str:
    await asyncio.sleep(1)                 # non-blocking wait -- YIELDS control
    return f"answer to: {prompt}"

start = time.perf_counter()
a, b = await asyncio.gather(               # launch both; their waits OVERLAP
    async_llm_preview("question 1"),
    async_llm_preview("question 2"),
)
print(a, "|", b)
print(f"took {time.perf_counter() - start:.1f}s")   # ~1.0s -- HALF the sync time

answer to: question 1 | answer to: question 2
took 1.0s


☝️ **Sync ~2.0s → async ~1.0s** for the exact same two calls. That's the payoff in one number: the two 1-second waits **overlapped** instead of stacking. The rest of this notebook unpacks *how* — `async def`, `await`, and `gather` — and then wires it into a LangGraph node. (Section 4 revisits this same benchmark once the keywords are introduced.)

## 2. `async def` and `await` — the two keywords

- **`async def`** makes a **coroutine function**. Calling it does **not** run it — it returns a *coroutine object* (a paused-able recipe).
- **`await`** runs a coroutine and **suspends** the caller until it finishes, *releasing control* so other coroutines can run meanwhile.
- You can only `await` **inside** an `async def`.

`asyncio.sleep(1)` is the async version of `time.sleep(1)`: it waits 1s but **yields** control while waiting instead of blocking.

In [3]:
import asyncio

async def async_llm(prompt: str) -> str:
    await asyncio.sleep(1)            # wait, but YIELD control while waiting
    return f"answer to: {prompt}"

# Calling it does NOT run it -- you get a coroutine object
coro = async_llm("hi")
print(type(coro).__name__, "->", coro)
coro.close()   # tidy up the un-awaited coroutine so Python doesn't warn

coroutine -> <coroutine object async_llm at 0x10c808940>


☝️ A coroutine is **inert** until something drives it. That driver is the **event loop**, started by `asyncio.run()`.

## 3. `asyncio.run()` — the entry point

`asyncio.run(coro)` starts an **event loop**, runs the coroutine to completion, and returns its result. It's the bridge from ordinary sync code into async land — your `if __name__ == "__main__"` for coroutines.

> **Jupyter note:** the notebook *already runs inside an event loop*, so `asyncio.run()` here raises `RuntimeError: asyncio.run() cannot be called from a running event loop`. Inside a notebook cell you can `await` at the top level directly (shown below). Use `asyncio.run()` in a real `.py` script.

In [4]:
# In a NOTEBOOK: top-level await works because a loop is already running.
result = await async_llm("what is asyncio?")
print(result)

# In a .py SCRIPT you'd instead write:
#     result = asyncio.run(async_llm("what is asyncio?"))
# We prove that form works by running it in a fresh thread (no loop there):
import threading
def run_in_script_style():
    print("asyncio.run ->", asyncio.run(async_llm("from a script")))
t = threading.Thread(target=run_in_script_style); t.start(); t.join()

answer to: what is asyncio?


asyncio.run -> answer to: from a script


## 4. The payoff — `asyncio.gather` runs waits concurrently

`await asyncio.gather(coro1, coro2, ...)` launches several coroutines and waits for **all** of them. Because each *yields* while waiting, their waits **overlap**. Two 1-second calls now finish in ~1s, not ~2s.

In [5]:
start = time.perf_counter()
a, b = await asyncio.gather(
    async_llm("question 1"),
    async_llm("question 2"),
)
print(a, "|", b)
print(f"took {time.perf_counter() - start:.1f}s")   # ~1.0s -- waits OVERLAP

answer to: question 1 | answer to: question 2
took 1.0s


☝️ Same two 1-second calls as section 1, but ~1s instead of ~2s. This is the whole point: when a graph makes several independent LLM/tool calls, `gather` collapses the total wait to the **longest single** wait. (Sync code got 2s; async got 1s. Scale that to 5 tool calls.)

## 5. Convert a **sync** LangGraph node to `async def`

A LangGraph node is just `state -> partial update` (Day 2). Making it async changes only two things:
1. `def node(...)` → **`async def node(...)`**
2. any I/O call inside becomes **`await`**ed

The **return shape is identical** — still a partial-update dict. LangGraph accepts sync *or* async nodes: call `graph.invoke(...)` for sync, `await graph.ainvoke(...)` when any node is async.

In [6]:
from typing import Annotated, TypedDict
import operator

class State(TypedDict):
    question: str
    messages: Annotated[list[str], operator.add]
    answer: str

# --- BEFORE: sync node (blocks while the LLM call waits) ---
def answer_node_sync(state: State) -> dict:
    reply = sync_llm(state["question"])          # blocking
    return {"answer": reply, "messages": [f"assistant: {reply}"]}

# --- AFTER: async node (yields while the LLM call waits) ---
async def answer_node(state: State) -> dict:
    reply = await async_llm(state["question"])   # await the I/O
    return {"answer": reply, "messages": [f"assistant: {reply}"]}

start = time.perf_counter()
print("sync  ->", answer_node_sync({"question": "q", "messages": [], "answer": ""}))
print(f"       took {time.perf_counter() - start:.1f}s")

start = time.perf_counter()
print("async ->", await answer_node({"question": "q", "messages": [], "answer": ""}))
print(f"       took {time.perf_counter() - start:.1f}s")
# NOTE: both ~1.0s -- ONE async call is NOT faster than one sync call.
# The win comes only when you run several concurrently (section 6).

sync  -> {'answer': 'answer to: q', 'messages': ['assistant: answer to: q']}
       took 1.0s


async -> {'answer': 'answer to: q', 'messages': ['assistant: answer to: q']}
       took 1.0s


☝️ Same return dict, same state shape. The only diff is `async def` + `await`. And note the timing: **both took ~1.0s** — one async call is *not* faster than one sync call. `async` doesn't speed up a single wait; it lets *other* work run during that wait. The speedup shows up in section 6 when two graph runs overlap. A node also stays **testable in isolation** — just `await` it directly, as above.

In [16]:
# Now run the SAME node 3 times -- watch the two styles diverge.
questions = [{"question": q, "messages": [], "answer": ""}
             for q in ["q1", "q2", "q3", "q4", "q5", "q6", "q7", "q8", "q9", "q10"]]

# SYNC: one after another -> waits STACK (3 x 1s)
start = time.perf_counter()
for s in questions:
    answer_node_sync(s)
print(f"sync,  10 calls sequential : {time.perf_counter() - start:.1f}s")   # ~10sec

# ASYNC: fired together via gather -> waits OVERLAP (max, not sum)
start = time.perf_counter()
await asyncio.gather(*(answer_node(s) for s in questions))
print(f"async, 10 calls concurrent : {time.perf_counter() - start:.1f}s")   # ~1sec

sync,  10 calls sequential : 10.0s
async, 10 calls concurrent : 1.0s


☝️ **There's the difference:** 10 sync calls = **~10.0s** (waits stack), 10 async calls = **~1.0s** (waits overlap). With 1 call both were 1s — nothing to overlap. The gap *is* the concurrency, and it grows with the number of calls: `N` sync ≈ `N × 1s`, `N` async ≈ `1s`.

## 6. `await` an LLM call inside a real graph

Wire the async node into a `StateGraph` and drive it with **`ainvoke`** (the async twin of `invoke`). Real LangChain models expose `await model.ainvoke(...)`; here `async_llm` stands in for that call so the notebook runs with no API key.

In [17]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
builder.add_node("answer", answer_node)      # <- the async node from section 5
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile()

# async node -> drive with ainvoke (await it), NOT invoke
out = await graph.ainvoke({"question": "What is await?", "messages": [], "answer": ""})
print("answer  :", out["answer"])
print("messages:", out["messages"])

answer  : answer to: What is await?
messages: ['assistant: answer to: What is await?']


### Why bother — two questions, concurrently
With an async graph you can fan out **independent** runs and `gather` them. Two questions that would take ~2s sync finish in ~1s.

In [18]:
async def ask(q: str) -> str:
    out = await graph.ainvoke({"question": q, "messages": [], "answer": ""})
    return out["answer"]

start = time.perf_counter()
results = await asyncio.gather(ask("What is a coroutine?"), ask("What is gather?"))
for r in results:
    print("-", r)
print(f"two graph runs took {time.perf_counter() - start:.1f}s")   # ~1.0s

- answer to: What is a coroutine?
- answer to: What is gather?
two graph runs took 1.0s


## 7. Test locally with `asyncio.run()` — the script pattern

Outside a notebook this is how you'd run the async graph from a plain `.py` file. We execute the exact snippet in a worker thread (which has no running loop) to prove the pattern works.

In [19]:
script = '''
async def main():
    out = await graph.ainvoke({"question": "run me from a script",
                               "messages": [], "answer": ""})
    return out["answer"]

# This is the real entry point in a .py file:
print("asyncio.run ->", asyncio.run(main()))
'''

def run_script_style():
    exec(script, {"asyncio": asyncio, "graph": graph})

t = threading.Thread(target=run_script_style); t.start(); t.join()
print("(in a real file you'd just write the async def main() + asyncio.run(main()))")

asyncio.run -> answer to: run me from a script
(in a real file you'd just write the async def main() + asyncio.run(main()))


## 8. Common async traps

| Trap | Symptom | Fix |
|------|---------|-----|
| forgot `await` | you get a `<coroutine object>`, not the result; `RuntimeWarning: coroutine ... was never awaited` | `await` the call |
| `time.sleep()` in async | blocks the **whole** loop — kills concurrency | use `await asyncio.sleep()` |
| `asyncio.run()` in a notebook | `RuntimeError: ... cannot be called from a running event loop` | top-level `await` in the cell |
| `invoke` on an async graph | may error / not await nodes | use `await graph.ainvoke(...)` |
| CPU-heavy work in a coroutine | still blocks the loop (async ≠ threads) | offload to `asyncio.to_thread(...)` |

In [20]:
# Trap #1 made concrete: no await -> you hold a coroutine, not the answer
maybe = async_llm("did I await?")
print("without await:", maybe)            # <coroutine object ...>
print("with await   :", await maybe)      # the actual string

without await: <coroutine object async_llm at 0x111380dc0>
with await   : answer to: did I await?


## 9. Recap + checklist

| Tool | Why it was used here |
|------|----------------------|
| **`async def`** | marks a coroutine — a function that can pause at `await` and let others run |
| **`await`** | run a coroutine, suspend the caller, **yield** control while waiting on I/O |
| **`asyncio.sleep`** | non-blocking wait (vs `time.sleep`, which blocks the whole loop) |
| **`asyncio.run`** | starts the event loop from sync code — the entry point in a `.py` script |
| **`asyncio.gather`** | run independent coroutines concurrently → total wait = the longest one |
| **`graph.ainvoke`** | async twin of `invoke`; use it when any node is `async def` |
